In [1]:
from pathlib import Path
import sys

# Ensure project root is on sys.path (notebook __file__ may be undefined)
NOTEBOOK_PATH = Path('/home/hmx42/Projects/Kronos_projest/my test/main/test01.ipynb')
PROJECT_ROOT = NOTEBOOK_PATH.resolve().parents[2]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from model import Kronos, KronosTokenizer, KronosPredictor

# Load from Hugging Face Hub
tokenizer = KronosTokenizer.from_pretrained("NeoQuasar/Kronos-Tokenizer-base")
model = Kronos.from_pretrained("NeoQuasar/Kronos-small")

# Create predictor (auto-select device)
predictor = KronosPredictor(model, tokenizer, device=None, max_context=512)
print('tokenizer, model 和 predictor 已初始化')

tokenizer, model 和 predictor 已初始化


In [2]:
# Initialize the predictor
predictor = KronosPredictor(model, tokenizer, max_context=512)


In [6]:
import os
import pandas as pd
import tushare as ts
from datetime import datetime, timedelta
from pathlib import Path

# Tushare token setup
token = os.getenv('TUSHARE_TOKEN', '72f0a12be0d24911ba9aa71f915040e0944fbcece2309d5adb5dded3')
if not token:
    raise RuntimeError("请先在环境变量中设置 TUSHARE_TOKEN，或者在代码中填写 token（例如：export TUSHARE_TOKEN=...）")
ts.set_token(token)
pro = ts.pro_api()

# Prediction window settings
lookback = 400
pred_len = 120

# Step 1: Get all A-share stocks (list of trading codes)
print("获取所有A股股票列表...")
stock_list = pro.stock_basic(exchange='', list_status='L')  # L=上市
print(f"总共获取 {len(stock_list)} 只股票")

# Step 2: Filter out ST stocks and stocks listed less than 60 days ago
print("过滤 ST 股票和上市不足 60 日的股票...")
cutoff_date = datetime.today() - timedelta(days=60)

filtered_stocks = []
for idx, row in stock_list.iterrows():
    # Skip ST stocks (name contains 'ST')
    if 'ST' in row['name']:
        continue
    
    # Skip stocks listed less than 60 days ago
    list_date = pd.to_datetime(row['list_date'])
    if list_date > cutoff_date:
        continue
    
    filtered_stocks.append(row['ts_code'])

print(f"过滤后保留 {len(filtered_stocks)} 只股票")

# Step 3: Fetch data for all filtered stocks
print("开始依次拉取所有股票数据...")
all_data = []
failed_stocks = []

end_date = datetime.today()
start_date = end_date - timedelta(days=int((lookback + pred_len) * 1.6))
start_str = start_date.strftime('%Y%m%d')
end_str = end_date.strftime('%Y%m%d')

for i, stock_code in enumerate(filtered_stocks):
    try:
        if (i + 1) % 10 == 0:
            print(f"  进度: {i + 1}/{len(filtered_stocks)} - 正在处理 {stock_code}")
        
        # Fetch data
        df = pro.daily(ts_code=stock_code, start_date=start_str, end_date=end_str)
        
        if df is None or df.empty:
            failed_stocks.append(stock_code)
            continue
        
        # Rename columns
        rename_map = {'trade_date': 'timestamps', 'vol': 'volume'}
        df = df.rename(columns=rename_map)
        
        # Ensure required numeric columns exist
        for col in ['open', 'high', 'low', 'close']:
            if col not in df.columns:
                failed_stocks.append(stock_code)
                continue
        
        if 'volume' not in df.columns:
            df['volume'] = 0.0
        if 'amount' not in df.columns:
            df['amount'] = 0.0
        
        # Parse timestamps and sort
        df['timestamps'] = pd.to_datetime(df['timestamps'])
        df = df.sort_values('timestamps').reset_index(drop=True)
        
        # Check if data length is sufficient
        needed = lookback + pred_len
        if len(df) < needed:
            failed_stocks.append(stock_code)
            continue
        
        # Keep only the most recent (lookback + pred_len) rows
        df = df.iloc[-needed:].reset_index(drop=True)
        
        # Add stock code column for tracking
        df['stock_code'] = stock_code
        
        all_data.append(df)
        
    except Exception as e:
        print(f"  错误: {stock_code} - {str(e)}")
        failed_stocks.append(stock_code)

print(f"\n拉取完成!")
print(f"  成功获取数据: {len(all_data)} 只股票")
print(f"  失败: {len(failed_stocks)} 只股票")

# Step 4: Combine all data and save to file
if all_data:
    combined_df = pd.concat(all_data, ignore_index=True)
    print(f"\n总数据行数: {len(combined_df)}")
    
    # Save to file
    data_dir = Path('/home/hmx42/Projects/Kronos_projest/data/processed_datasets')
    data_dir.mkdir(parents=True, exist_ok=True)
    
    csv_path = data_dir / 'all_stocks_data.csv'
    combined_df.to_csv(csv_path, index=False)
    print(f"所有数据已保存到: {csv_path}")
    print(f"文件大小: {csv_path.stat().st_size / (1024*1024):.2f} MB")
    
    # Display summary statistics
    print(f"\n包含的股票数: {combined_df['stock_code'].nunique()}")
    print(f"数据时间范围: {combined_df['timestamps'].min()} 到 {combined_df['timestamps'].max()}")
else:
    print("没有成功获取任何数据!")

获取所有A股股票列表...
总共获取 5517 只股票
过滤 ST 股票和上市不足 60 日的股票...
过滤后保留 5222 只股票
开始依次拉取所有股票数据...
  进度: 10/5222 - 正在处理 000017.SZ
  进度: 20/5222 - 正在处理 000030.SZ
  进度: 30/5222 - 正在处理 000045.SZ
  进度: 40/5222 - 正在处理 000063.SZ
  进度: 50/5222 - 正在处理 000099.SZ
  进度: 60/5222 - 正在处理 000400.SZ
  进度: 70/5222 - 正在处理 000415.SZ
  进度: 80/5222 - 正在处理 000429.SZ
  进度: 90/5222 - 正在处理 000514.SZ
  进度: 100/5222 - 正在处理 000528.SZ
  进度: 110/5222 - 正在处理 000539.SZ
  进度: 120/5222 - 正在处理 000552.SZ
  进度: 130/5222 - 正在处理 000564.SZ
  进度: 140/5222 - 正在处理 000581.SZ
  进度: 150/5222 - 正在处理 000598.SZ
  进度: 160/5222 - 正在处理 000620.SZ
  进度: 170/5222 - 正在处理 000636.SZ
  进度: 180/5222 - 正在处理 000665.SZ
  进度: 190/5222 - 正在处理 000685.SZ
  进度: 200/5222 - 正在处理 000702.SZ
  进度: 210/5222 - 正在处理 000716.SZ
  进度: 220/5222 - 正在处理 000727.SZ
  进度: 230/5222 - 正在处理 000751.SZ
  进度: 240/5222 - 正在处理 000767.SZ
  进度: 250/5222 - 正在处理 000788.SZ
  进度: 260/5222 - 正在处理 000800.SZ
  进度: 270/5222 - 正在处理 000815.SZ
  进度: 280/5222 - 正在处理 000831.SZ
  进度: 290/5222 - 正在处理 000860.

In [7]:
import time, torch
model.eval()
torch.cuda.empty_cache()
start=time.time()
torch.cuda.synchronize()
pred = predictor.predict(df, x_timestamp, y_timestamp, pred_len, sample_count=1, verbose=False)
torch.cuda.synchronize()
print("elapsed", time.time()-start)
print("peak mem (MB):", torch.cuda.max_memory_allocated()/1024**2)

elapsed 1.8525490760803223
peak mem (MB): 123.33740234375
